# Adam and AdamW by hand

Write the Adam optimizer yourself, with both kinds of weight decay, so that it matches PyTorch step for step.

```python
class MyAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0, decoupled=False):
        ...

    def zero_grad(self):
        ...

    def step(self):
        ...
```

- `params` is an iterable of tensors with `requires_grad=True` (for example `model.parameters()`, which is a generator:
  store it as a list).
- `zero_grad()` clears the gradient of every parameter (set `p.grad = None`, as PyTorch does).
- `step()` updates **in place** every parameter whose `.grad` is not `None`, and skips the others. For a parameter $\theta$
  with gradient $g$, keep a step count $t$ and averages $m, v$ (zeros at the start) **for that parameter**, and do:

$$\begin{aligned}
&\text{if decoupled:}\quad \theta \leftarrow \theta\,(1 - \eta\lambda) \qquad\qquad \text{else:}\quad g \leftarrow g + \lambda\theta \\
&t \leftarrow t + 1, \qquad m \leftarrow \beta_1 m + (1 - \beta_1)\,g, \qquad v \leftarrow \beta_2 v + (1 - \beta_2)\,g^2 \\
&\theta \leftarrow \theta - \eta\,\frac{m / (1 - \beta_1^t)}{\sqrt{v / (1 - \beta_2^t)} + \epsilon}
\end{aligned}$$

where $\eta$ is `lr`, $\lambda$ is `weight_decay` and `betas` is $(\beta_1, \beta_2)$. With `decoupled=False` this is
`torch.optim.Adam(params, lr, betas, eps, weight_decay)`; with `decoupled=True` it is `torch.optim.AdamW(...)`. With
`weight_decay=0` the two are the same.

**Example.**

```python
p = torch.tensor([1.0, 1.0], requires_grad=True)
opt = MyAdam([p], lr=0.05)
p.grad = torch.tensor([50.0, -0.002])
opt.step()
p           # tensor([0.9500, 1.0500], requires_grad=True): Adam's first step moves each entry by about lr
p.grad = torch.tensor([50.0, 1.0])
opt.step()
p           # tensor([0.9000, 1.0129], requires_grad=True)
```

The grader trains small networks with your optimizer and with PyTorch's from the same start, and compares every
parameter after each of up to 30 steps (relative tolerance about $10^{-5}$).

**Restrictions.** Only `import torch` (and `import torch.nn as nn`). Nothing from `torch.optim` (the grader bans the
name `optim`). Update the parameters in place inside `torch.no_grad()`, so that the model's own parameter tensors
change; do not replace them with new tensors.

**Constraints.** Up to 20 parameter tensors with up to $10^5$ entries in total; `lr` in $[10^{-4}, 0.1]$; any $\beta_1, \beta_2$
in $[0.5, 0.9999]$; `eps` in $[10^{-10}, 10^{-4}]$; `weight_decay` in $[0, 0.5]$. Time limit 1 s per `step()`, CPU.


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `problem3_submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile problem3_submission.py
import torch


class MyAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0, decoupled=False):
        pass  # Your code

    def zero_grad(self):
        pass  # Your code

    def step(self):
        pass  # Your code


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import problem3_submission as submission
importlib.reload(submission)
# e.g. submission.MyAdam(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNq9W41z2zaW/1dw7syaTGVGVpJ2V6l757ZuNzf5mjg77Y2tYSERkrmmQIagIrte/+/3fg8ACUpyHKd7l8zIEgi8L7xvgDd7ebY3Fnurw8ODavhkbyD2lqqRNHSz/aTJm0Jh7DiTSyF1JvDlVzG9Fhf0C1MKZUyp/bpi+C1DzPWqUYZGR0P6meXzeT5bFc01jTylAaWbGt/3Xl0DIC8ps1WhUi2XjLCqy2mhlk9Ss5ouc2NywkGzZFGUa5Wl+bIq6wYIzvaasp5d7E3o6VRqrTIeLKsmX/rBtCjLCpPnsjAKfCnTpDRBlasmxYMnILORCwtR64TXJ9IR1xtYs2SANNFluqhlxnhMuapnTHtZ54tcywLzKlk3Ixps6pW6pd/zvGC53OxdyFqT7JKKBXHO//+uikrVRjQXshFOBAIYMGh5FtHv87pcCrfcjz76PU7O9bk+dtNFbsSs1A1J+vFK5415/B3+fP/YQaXfVZ59/9jOJiIS8aIRmZrnBFVAPmK+0rOG5G4ENiXjwfSRpa2Rl4q+qXPd7Y+wWyh+p6HfE3Hy2/Grty9PTkWRmwYsKQsWwIwwF+VaC1r0bqVF9DE3OVEVPxenANeIekVo1UdVX/MiYu1nmRermlbWMjdK4KdY582FkILgGblQYl3nDTEs5mUtflB6LIy8FmuQu5aGnpZ6MRAaUEW2WlbiIs8ymp7ratUYSM/ugnYinZXVdfsDygL5up+a1l8LgqorjH4lfiwLORU/vv2HEbJWwkBNa8hKg5R9I16NxNu6HAhTiqKcyQIQc70wYrkioUwVhEoiKmEpOT0ucsiBDK25oK2cyxlpXAJMo2SIzZViscL+RxazIViFOmguaiUzYSpFG0az6nK1uChgrcWc7BdUzMoagq4VhN6IfA4C6ScblOPDmQaEcvr25OSn9OfjH9+/eSeOgB0Mn+tZIQk7tiE6uZqpCpoSj8+1oH8VPbKzSKFI4iqLSBezgViahZ9DeHUJIevMjeBft7kR5nZAUl1FV34tOY9gjd8hWGQwOif+c20aqWcquhrY58l7pU1Zx8FyRquaVa3FVZKRK5xdRHEyq1b0ydscxXayYjbFC0Z3UtdlHUCxLAewdJVII+taXhPdHR/mQlYqWpTNgJRS45P08+h8jzSb/OP5nqfMkKybVVWoCJzT/DjhpY4WlinN8ZMAKx6I+fneDQDe0oYbi0vcGPI76qpSs4Z04iaYf0voOspmRWk+RdlA1E1ZHB2qg28GQrqvf/X0kmWtieSWWGmaa2J0XpQShGGccfYfBMwsLH9gaW2/3sGOZcBNj3fx5h8xey0C2pDckHvLG1pNdBRFFAvyFNDC8NnaPQvRw5PKnD2hxppcz1vgTpGx30XhhEjCcOLChxMXPkLFU3VNEmNBgLilvMIfOTXRQhyIdRzHgE2M5n8ooShwiSGMb4ep9OXEjk58lAX5BxEVsl7A7SoorLihP+PkyeI27u29+rCSxWdoZd+o7FQQrTPW9EHMEiUiZNPU7vn5Hm9HB2SHugQK0s35P1eLUDWY/tTJgYjbMqZQqi0E7Mp4k2JSBlDLzHcwyBnf0KP/qHukYRYN9TYjNymbCG8Hvpk794MRBjvSLdmiH6MkCgc4TlJOtdK0R86C3B92zmHd95P2PdQ+panRsqJI3vz/uuai1OqBTplQeq28gkpybN+LOaWF9fbpibJ8RoJE4hKH9PSZ61EHgFGmVHX0njK9QMEcge9JfJvkbQOIeyEEQwlg8rOrnuBpWyI5ENMvl7v8LLnbKdYuGN+Xil1C7AzH9F2BQyQtEhNN412+hpZ3fmbn8tZPTN1X3tu+ZYODgfV2pNk63CsPhyEEyUuZLldkFJTdRHM9EI/InZJlPHp0ucY3T4lNHX+kCCDmmjHPkaCCB8pxKAtckHlJTSljvVgtKSmnECKqQs5U4rNOjvqwJ/KLZ51lyZjzWYkFwDixMyk7Q/DQ0SZB9jGW5KTIxLCJsVRRKqNqSfHtj7yK7BrG0VNw9ietcjknck2FDelippDg5/McmSVS5o6Xr25y8bU4vH0u1mV9iaxesvIKbB/lo62/dEIm4jsJI4nPdWQUUkGi6nOl/DwUMXnXMOXekUlz5Ujx0FDuTRxAQJzr7rcrKXm3NRRQvSEQNUowKpCWBixxHv5KzsSMEGAJMWmzblQrnNgjOaDq4mC6ysiTYphzcErLcxRQq4rMSTTrfKYAkCSUU4Qu586CiWYvUcMqhJE55UsI34BAK1b0LBHv8SCvaZiYKfaBAnHBIoOEVdZTq2aIbJIgJFRdzlMGo2rvbe5RpazZvZhylMalI47fI+E2UTwWYdnQqWQKVaxhDdEo1DvaRELz3ZGDtOGKplTTXHZDcjQQlyPC1vePTPJgY7DHyf2iYDohiBGLYRQMsxioYosyhNg7xBHmmyFDbEdNWbLeUPGfNeNkNL+lLJ+0TD13VVerxKxMZHYMSKw06vkbJxq3jmtIKuz8Fu/4xyipsqQyeE6qtUACa4vMG7dPDAjW8UkDrfWCrFNlfsO6Aod2MiuXJO25JPVL24nn3KtZGSlz2yBRvS7HLxghCyDKuuZBzk7jIi8yWN0MnREqmHPYAmb+9+mb10SrZssxTUYUsq1W180FjRy8EB6P+M61ONIsr78X33U46CH9Jqv8F48132+V/GRsQflPG3vVFPm0HbJ/aCRZNXnRDZft13+asmsSVLK5CJebfKFlt8xcm432gv9RU1yYytllO0JarNEssL0GM8uJTa1ad0cDl3lzUCiaJg6Tb5JDl9OZMfpES/ToBuxSqLpwwcc7v0rmJOXFufY4knlekEb7n9H5HpFd1goVoOu0UCaaPPqHvtTo4ZiyQEel5A4AIcwr7FryyGWKfz95d0KW42SRvKW/UZqiDZamcUKOC8spq6tkTZSda5JKgskJaauqm2hIxkDpA8DEBHBX74t7QeIritUf5FicPB2OuBP28tfj/zlNfzh+/frkJ6Lg5nwvdX3DNAUzxJjfT/tTXamZ+0Z5vv02K5cV0Xq+d3uuX7558zZ9/eank1MCF5GuJD+XZKv48isprrJfj821nrUPXlIO+SPBsL9OVfDjJ0oyu1+/KI34XNYnV1Uctlje217MfV2WVFKlt4ygY6sl+RsKW6q1WC4VPaCwG4HMZpbOLtTs0kSmnmGLG9mPt++svSMbpmAFOZaXVjhLsyDBPCe3iAzQ9vjYbEyz3+4OJ0OulYhA1dS5bSv2QpQNQmeTXdlsUytK7FhKpCVUXxOh/Tz09Jrq8yvOQ9GUU9v54Zkje0zawj1gR/0YbtLwclcil9r6mRuV4K8ub8f4TrOp+Ll1BLo+NEICySshx0x2stGchozOJo7QqWzHOU42PuRwOzRdGQYWjE/JHlLbK3XjUHFKGcl284xMBZlYbRMFC5gg2PkRKhok3fkUXXhi55JzN1UkUGyHgHvigcyXslmuinBkzt0RctXwzST8tSwuI2zGRugO0nRMt/psS4PNqqJNZTExYXLH2wHMtVQoX44kT0JSvkRrwf5MCBsJEpljtKTMk9wRijkAXzKxdiPi8e7YGOxFIrPM4Yi3J3d7E8xLTFXk2G7gPDs4nOxY2G1esqoyZN071oZFYvFpMf5MXm+TG9pQ2iuWo2u8E/udQe2QJa9wgqTvf0KKmxIkaBtSCETXk8AuDdhYepfwvmgp8/lQoR9729nkfUMhmA4Y2mdBfR145E8BzLO7udo9aRfGLlht4mTDT2RVKZ1FvuNjF53vWZdn3ft/9uW1CwtHwR9y/cZFsePV4tggBsW2t7KxICndvFeyeUUp45ZOWx/0QOKIsEAhA5BIEh2oO92/zwKslViVxygbBOYl/yxRnrKDjQI8cbzbu3yNiEKC8aBu9gdi3wLxFgUD3Cd8VPcu9tGKvd3dT9xFP1oWO8g3nvYO2FftCeJUagPZzi5chOBUsAsQ3H5WkorY3/hv/NxWo8VaXpsDe6pKlQzlhXkLCErpoJW6uB54nFS5dKEmslpgm0gulULScS0KtaDSfI20SfDMJEniIJhRUiayEptiAV/kXF/aXYiCoPkXDo5BFLbkuuBLavgvEQWx9C+ilxcGKkQYHqw6RIERTj6AH5l4h+4Q5PjP7LAuPY72SLbbZaK8z3xqjSgtK5tP9rttmG6NbMP0PodhVJSkF/8lPGyUrIHdiMhmTg7D2XASKvcO9h8igg7rp5hn57bFMo9+GcM2TWJl7zFKwzBjKHWtLpTmIrbNHXnVv5d/x9ntJyr1eZ0TmOI65Uw2IhBU6KaopvoZ/RutDpjOTJlZnXNVISpSVbLvhXAn0Twh98fXKNjIXBXmGT6s73eYpmir+Lo1obqZvjdpM41UkqbtAyr5XLIJ4JRpzjmoz4GHQNBOzVF+Kp/zofJrWXBZ6RotEzSs4GW5N+mUjj6RjgVZe28IW8EKg9l8nNaR74SJExM+IVHB+Qiy/9sbxnqL+V7YS8rEo7YWI0+7WeWikpX14uNZmyJ6Vu6cOZoEJbGTFBLwI9FOeeLTddJ7GkfDISlKmZkoYiIeQ21gE9yK2AM8ilnoZETe2VH5BIiOmHCClwZ3E4/Etl3YghFDZxP8Rqlnup+2MWNV1lXS0EQMvC5pC0L4Zx20Cci5oxRtDR0ZLEzQnFmqJhwt8IutZBKms76k8/ms4b7jFtrQS3DXImJx4nKIiez0MMpbLfEtzzkRnZfJaYP2yYs3Xni2yZPYP5H7dfril+OX714NXJXuppI7S7m560+BA1/Wv6EEWY6GnpZ+dcyXYLpmFW1nlteKjM9uRkSUbqZZnkbV5EBRezJfvH/x6uRd+u7k+OWgpW4jzTGVgvb0W2EJRlP0ZmxnB91JPjIJOAoueFnV6BpztE2Bq9rAt5puo3OwGCFQR/jYXEgWY+eZMzzmgmFiFX+bJbYhVSfoAaV2GTxP/AWCG/Z7E67t0mtJOEV0xjGx3szyCA/MXo09blOWl3ymIG78foyTITrA3nU5LD9Io9oOke2BdC2xH16eDIeHHQEPYuMuit3BEJOJkxjcECvdRrkk6BPxqDXrLdC7BOX9CvCSPkOj+CQ+ogL86XA4HE++2JD5KmK/j8ND4f0OO8efFfsTTeJlYB/F929uJ6usVDZpspftKLoAxG1Ytz+QhcXDbPJ8z176gxW2IaO9B9jy7e4R3m97i8D4FnfZkoXmZrnrfq6x5as8OwWewV8d9Em8c3uINUgaIipU5tax4xt8+0dZG48jyRu1NBHfntH9qMBe1Z/74wBFEhnRPA4bXuQlcAjJKQkwbiTPHI6h/fWK/JZVCmQrfItAe9Y2HC7cc65XKty/mQ2w1h+OGcjZs/GE/DefCxO1rkst2D2e7znQfnKADU/RDUbZs6tMcf7WHvpgaJgMbx9yJrZ98eELos6fjTzucG7bK3+RS3ObcNZJDpaKxHv77saWB+8W29zD2jg766LUC38us+mzRZRre8uMs/kBCoi5rIU/GYxDJ+CQ8+HGRk/7TzLc0oz0esddlQfFkn8XLeE9Az6vyD4zhnSwWgUHvJq0OIvuOqgdiCe7wptLZye+IKtbx/bgUNQusLnqEUrHqMtdP5WVslOZlmURbRFmnwFU09NcgGt64NoVvkC5M6Twtea58AWP9W5pivoGB2VjfzyAcsee7LbBAme6O29L24tH4a9EsyppffcZ3oDPuwfuRgjIau+Qk9d33ltdyWUFLzgJbxg3fPBMupH/ocxR9GwgDr8ZiL9im9sbUkzFUmpc/jF8f9aeVXNPVF7jMKV/9oHbQpApbsww4LMxFZMOydnhuFc/OAhfEwitk5cU2WVtb0wRWzp5L/VFFE/6J+g6OVUfVjjLoVLhkYXAOIIjukw20jGnj56S4mZHxN3l0ROiA1dZjg6TYXsJF36c2WxPE6P4LpZ7F7xwkq8RVwnNwq89WsTikcUyCOZBkYiOS1z9HMT9+cF14nKJI90UN/ltqkTBVtEeu41qVGUIyNpBTmcFbiRd5tURKsWAOXeRdqQOnnX3jr/pNzTe1zgWa9Zld6mGexiGIzPuX+AOAccs+8YJm1E7ZnnjNzyee8KFnJO7cO8ggNpevwOlt9c6f2C3fSPF2x5m4/g0kjjAJJoIsmnL8TxjxcuzqLJJTcWXw6S/6VU1KQC4e1IsEZJnYhkhkfRg2us7PmO6xCVbXGaMwrGkKitKLzJyl7gZy71SCD3uEE69JmFfoukWiqaF99tAIH1u1RQXwYZDv3/8GdxJgyC7O0CHTg+wJrSluT2ggrYQMZgfwZRYEoQ+mtrv03gzxaDR5A9V20sn0eaRlE58/5Ri06wujUmRe2OrGFv0GwG/jhN0qtay3gJAThIK6pueEJntQVzCQRAjuzIehnw2nCRrlS8umgSUkbx48da/r+wts1baZFsNkHEynuPmBj8Hrg7TVJG8uJtWbV2S7fSpryaTfv/A6xVLOGFWgh41RM2D0ebF6U2l3UCCUJLznb7zvRs2/9sx2yULZT9g1Ig1unou983w3otWa0rAcTXVPBf2NA9rl+3Nyd49buwNdIswHm6esYFS3z0K77NeUappicc37+mtODftKo4HO457Qr6sgMjabJVoL33exW3kGKpabqiUTBaJqBKkN/Y0xF786r1pBqcRsg3iI1uyVKS8HzwT9sQ0S/t227fjTW11J8UWX/suA2WpH+5/l6H/TkNUUZL1IeZ3GWJ+scEWZdGOxH3jNQYvTet9eUtv8Hk7CKziBsx98csOYRrB1YTv4bbR06pddMY2QR8U82tSGSpwDO9CeG8YDuoocMjRWUXTi/qIaqxnPgB4q++DfzYE/AP6HI4mHbieqbHybm5Ja+aDDYjD5G/PmOJnk7gLls/sPV7HshXq4VjgRYWbwGXQbMSo3rsTIcRW8T7JDsR1By8fvkjAtZq36+xLmFbMH7bEDGNYcLToi3YgQtoCrf2wk5EFvx03f9A2fAi++9ekgnTl2x07MHI7cGYrlY8D8dQ6o4/sE7b3ZbK5MczQ4ehvk2315ntCKd5WTUv6m1KugqvZobpvJ2hUUPD1IptXFDXaBIdIDihO4wVZTiSsyA9b1eu2ZJOG2co05TKdEh8mpTiZUqS/jwCeLKJhQqk7cfg3QoP8wL501uT6uo2EtieOtvguxxyQOqRMmcEebYK1ieQ2I23yqQ6ebjJVjFIbyFNKn+T1ffxwrsk5ZriKaDgUUZt/HXHHJgZDROvo2T0MEdMhMBqi/dkEdv/28KvMn0P/rzsZGIUMWHMl+p9+Ef2jkH6GtU3+r1sq3oayFPThnKTVjfvYspfnueDaN44aDidhqsUXjaF9RN4TTvS+BYtwh6N7WDzcYnH47HN4dBVQIZfTTAozti2CaMSK8W28KYIu1Q34XbrS5Ju7AtRyI5v3Fr2dz3/rvd+nMue7suadubjNxYoicjGEklnOhN17mHY0QbIWZJV9grkXGgC2L1DPcPnZVWvtbNpbv5vsIvdub/8XJZbXxw==")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
